In [1]:
# Import libraries

import pandas as pd 
import numpy as np
import wrds
import os # check if cached parquet files exist (skip WRDS re-download)

In [2]:
# Connect to WRDS - enter credentials

db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
# Get Compustat Annual Information

funda = db.raw_sql('''
                   SELECT  c.gvkey, c.cik, sic, naics, datadate,  fyear,
                           sale, revt, cogs, xsga, dp, xrd, xad, ib, ebitda, ebit, nopi, spi, pi, txp, ni, txfed, txfo, txt, xint,
                           capx, oancf, dvt, ob, gdwlia, gdwlip,
                           rect, act, che, ppegt, invt, at, aco, intan, ao, ppent, gdwl, fatb, fatl,
                           lct, dlc, dltt, lt, dm, dcvt, cshrc, dcpstk, pstk, ap, lco, lo, drc, drlt, txdi,
                           ceq, scstkc, emp, csho,
                           prcc_f
                   FROM comp.company AS c 
                   JOIN comp.funda AS f
                   ON f.gvkey = c.gvkey
                   WHERE datadate >='1975-01-01'
                   AND indfmt='INDL'
                   AND datafmt='STD'
                   AND popsrc='D'
                   AND consol='C'
                   ''')

# can add alias c. and f. for readibility, but only kept for gvkey and cik since in both columns (ambiguous). cik, sic and naics are from comp.company 
# use industrial format, standardized date format, domestic (North America) companies and consolidated level
                          

In [4]:
# Clean Compustat Annual Information

funda['sic2'] = funda['sic'].str.slice(0,2) # give the BROAD industry (recall slice use (included,excluded))

funda['sic'] = pd.to_numeric(funda['sic'], errors='coerce') # bad values become NaN instead of crashing

funda['prcc_f'] = funda['prcc_f'].abs() #  if no trade occurred a day/month, CRSP instead reports the average of the bid and ask quotes, with the value negated to flag that it's not a real trade price

funda['mve_f'] = funda['prcc_f']*funda['csho'] 

funda = funda.dropna(subset=['at','prcc_f','ni']) # remove noise and avoid a lot of NaN when computing

funda['datadate'] = pd.to_datetime(funda['datadate'])

funda = funda.sort_values(['gvkey','datadate']) # sorting for shift() and cumcount() reason

funda = funda.drop_duplicates(subset =['gvkey','datadate']) # keep only one row per firm/datadate. keep='first' is the default. can remove it and everywhere else. subset= is non-mandatory

funda['count'] = funda.groupby('gvkey').cumcount() + 1 # create an firm counter (default cumcount() start with 0 - thus here starts with 1)

# several of these variables have lots of missing values and can result to a lot of NaN
# a missing part counts as 0 (x + NaN = NaN otherwise); NaN only if both are missing

funda['dr'] = np.where(funda['drc'].notna() & funda['drlt'].notna(), funda['drc'] + funda['drlt'], 
                       np.where(funda['drc'].notna() & funda['drlt'].isnull(), funda['drc'],
                                np.where(funda['drlt'].notna() & funda['drc'].isnull(), funda['drlt'], np.nan)))  
funda['dc'] = np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].notna() & (funda['dcpstk'] > funda['pstk']), funda['dcpstk'] - funda['pstk'],
               np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].isnull(), funda['dcpstk'],
                        funda['dcvt']))

# missing interest / SG&A expense set to 0, only for operprof (Fama-French convention)
funda['xint0'] = funda['xint'].fillna(0)
funda['xsga0'] = funda['xsga'].fillna(0) # Prof. Green agreed of this corrected version

In [5]:
# Create simple-just annual Compustat variables
# could make it computational easier with unique shift() per variable lagged but did not for readibility of variable creation (doing it later in the code)
# like lag1 = funda.groupby('gvkey')[['ni','at','dltt','act','lct','sale','cogs']].shift(1)

funda['bm'] = funda['ceq']/funda['mve_f']
funda['ep'] = funda['ib']/funda['mve_f']
funda['cashpr'] = ((funda['mve_f']+funda['dltt']-funda['at'])/funda['che'])
funda['dy'] = funda['dvt']/funda['mve_f']
funda['lev'] = funda['lt']/funda['mve_f']
funda['sp'] = funda['sale']/funda['mve_f']
funda['roic'] = (funda['ebit']-funda['nopi'])/(funda['ceq']+funda['lt']-funda['che'])
funda['rd_sale'] = funda['xrd']/funda['sale']
funda['rd_mve'] = funda['xrd']/funda['mve_f']
funda['agr'] = funda['at']/funda.groupby('gvkey')['at'].shift(1) - 1
funda['gma'] = (funda['revt']-funda['cogs'])/funda.groupby('gvkey')['at'].shift(1)
funda['chcsho'] = funda['csho']/funda.groupby('gvkey')['csho'].shift(1) - 1 
funda['lgr'] = funda['lt']/funda.groupby('gvkey')['lt'].shift(1)-1
funda['acc'] = np.where(funda['oancf'].notna(), # np.where like ifelse on R
                        (funda['ib']-funda['oancf'])/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2), 
                        ((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2))
funda['pctacc'] = np.where(funda['oancf'].notna(),
                           funda['ib'] - funda['oancf'],
                           ((funda['act']-funda.groupby('gvkey')['act'].shift(1)) - (funda['che']-funda.groupby('gvkey')['che'].shift(1)))- ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1)) - (funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1)) - (funda['txp']-funda.groupby('gvkey')['txp'].shift(1)) - funda['dp'])) / funda['ib'].abs().replace(0, 0.01)
funda['cfp'] = np.where(funda['oancf'].notna(),
                        funda['oancf']/funda['mve_f'],
                        (funda['ib']-((funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'].shift(1))-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txp']-funda.groupby('gvkey')['txp'].shift(1))-funda['dp'])))/funda['mve_f'])
funda['absacc'] = funda['acc'].abs()
funda['age'] = funda['count']
funda['chinv'] = (funda['invt']-funda.groupby('gvkey')['invt'].shift(1))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['hire'] = np.where(funda['emp'].isna() | funda.groupby('gvkey')['emp'].shift(1).isna(),
                         0,
                         (funda['emp']-funda.groupby('gvkey')['emp'].shift(1))/funda.groupby('gvkey')['emp'].shift(1))
funda['sgr'] = funda['sale']/funda.groupby('gvkey')['sale'].shift(1) - 1
funda['chpm'] = (funda['ib']/funda['sale']) - (funda.groupby('gvkey')['ib'].shift(1)/funda.groupby('gvkey')['sale'].shift(1))
funda['chato'] = (funda['sale']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)) - (funda.groupby('gvkey')['sale'].shift(1)/((funda.groupby('gvkey')['at'].shift(1)+funda.groupby('gvkey')['at'].shift(2))/2))
funda['pchsale_pchinvt'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['invt']-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['invt'].shift(1))
funda['pchsale_pchrect'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['rect']-funda.groupby('gvkey')['rect'].shift(1))/funda.groupby('gvkey')['rect'].shift(1))
funda['pchgm_pchsale'] = (((funda['sale']-funda['cogs'])-(funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1)))/(funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1)))-((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))
funda['pchsale_pchxsga'] = ((funda['sale']-funda.groupby('gvkey')['sale'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))-((funda['xsga']-funda.groupby('gvkey')['xsga'].shift(1))/funda.groupby('gvkey')['xsga'].shift(1))
funda['depr'] = funda['dp']/funda['ppent']
funda['pchdepr'] = ((funda['dp']/funda['ppent'])-(funda.groupby('gvkey')['dp'].shift(1)/funda.groupby('gvkey')['ppent'].shift(1)))/(funda.groupby('gvkey')['dp'].shift(1)/funda.groupby('gvkey')['ppent'].shift(1))
funda['invest'] = np.where(funda['ppegt'].isnull(),
                           ((funda['ppent']-funda.groupby('gvkey')['ppent'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1)))/funda.groupby('gvkey')['at'].shift(1),
                           ((funda['ppegt']-funda.groupby('gvkey')['ppegt'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1)))/funda.groupby('gvkey')['at'].shift(1))
funda['egr'] = (funda['ceq']-funda.groupby('gvkey')['ceq'].shift(1))/funda.groupby('gvkey')['ceq'].shift(1)
funda['capx'] = np.where(funda['capx'].isnull() & (funda['count']>=2),
                         funda['ppent']-funda.groupby('gvkey')['ppent'].shift(1),
                         funda['capx'])
funda['pchcapx'] = (funda['capx']-funda.groupby('gvkey')['capx'].shift(1))/funda.groupby('gvkey')['capx'].shift(1)
funda['grcapx'] = (funda['capx']-funda.groupby('gvkey')['capx'].shift(2))/funda.groupby('gvkey')['capx'].shift(2)
funda['tang'] = (funda['che']+funda['rect']*0.715+funda['invt']*0.547+funda['ppent']*0.535)/funda['at']
funda['sin'] = np.where(funda['sic'].between(2100,2199) | funda['sic'].between(2080,2085) | funda['naics'].isin(['7132','71312','713210','71329','713290','72112','721120']),
                        1,
                        0)
funda['act'] = np.where(funda['act'].isnull(), 
                        funda['che']+funda['rect']+funda['invt'], 
                        funda['act'])
funda['lct'] = np.where(funda['lct'].isnull(), 
                        funda['ap'], 
                        funda['lct'])
funda['currat'] = funda['act']/funda['lct']
funda['pchcurrat'] = ((funda['act']/funda['lct'])-(funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1)))/(funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1))
funda['quick'] = (funda['act']-funda['invt'])/funda['lct']
funda['pchquick'] = (((funda['act']-funda['invt'])/funda['lct']) - ((funda.groupby('gvkey')['act'].shift(1)-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['lct'].shift(1))) / ((funda.groupby('gvkey')['act'].shift(1)-funda.groupby('gvkey')['invt'].shift(1))/funda.groupby('gvkey')['lct'].shift(1))
funda['salecash'] = funda['sale']/funda['che']
funda['salerec'] = funda['sale']/funda['rect']
funda['saleinv'] = funda['sale']/funda['invt']
funda['pchsaleinv'] = (((funda['sale']/funda['invt'])-(funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['invt'].shift(1)))) / (funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['invt'].shift(1))
funda['cashdebt'] = (funda['ib']+funda['dp'])/((funda['lt']+funda.groupby('gvkey')['lt'].shift(1))/2)
funda['realestate'] = np.where(funda['ppegt'].isnull(),
                               (funda['fatb']+funda['fatl'])/funda['ppent'],
                               (funda['fatb']+funda['fatl'])/funda['ppegt'])
funda['divi'] = np.where((funda['dvt'].notna() & (funda['dvt']>0)) & (funda.groupby('gvkey')['dvt'].shift(1).isnull() | (funda.groupby('gvkey')['dvt'].shift(1)==0)), 
                         1, 
                         0)
funda['divo'] = np.where((funda['dvt'].isnull() | (funda['dvt']==0)) & (funda.groupby('gvkey')['dvt'].shift(1).notna() & (funda.groupby('gvkey')['dvt'].shift(1)>0)), 
                         1, 
                         0)
funda['securedind'] = np.where(funda['dm'].notna() & (funda['dm'] != 0), 
                               1, 
                               0)
funda['secured'] = funda['dm']/funda['dltt']
funda['convind'] = np.where((funda['dc'].notna() & (funda['dc']!=0)) | (funda['cshrc'].notna() & (funda['cshrc']!=0)), 
                            1, 
                            0)
funda['grltnoa'] = ((funda['rect']+funda['invt']+funda['ppent']+funda['aco']+funda['intan']+funda['ao']-funda['ap']-funda['lco']-funda['lo'])- (funda.groupby('gvkey')['rect'].shift(1)+funda.groupby('gvkey')['invt'].shift(1)+funda.groupby('gvkey')['ppent'].shift(1)+funda.groupby('gvkey')['aco'].shift(1)+funda.groupby('gvkey')['intan'].shift(1)+funda.groupby('gvkey')['ao'].shift(1)-funda.groupby('gvkey')['ap'].shift(1)-funda.groupby('gvkey')['lco'].shift(1)-funda.groupby('gvkey')['lo'].shift(1))- ((funda['rect']-funda.groupby('gvkey')['rect'].shift(1))+(funda['invt']-funda.groupby('gvkey')['invt'].shift(1))+(funda['aco']-funda.groupby('gvkey')['aco'].shift(1))-((funda['ap']-funda.groupby('gvkey')['ap'].shift(1))+(funda['lco']-funda.groupby('gvkey')['lco'].shift(1)))-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)

rd_cond = (((funda['xrd']/funda['at']) - (funda.groupby('gvkey')['xrd'].shift(1)/funda.groupby('gvkey')['at'].shift(1))) / (funda.groupby('gvkey')['xrd'].shift(1)/funda.groupby('gvkey')['at'].shift(1))) > 0.05
funda['rd'] = np.where(rd_cond.fillna(False),
                       1,
                       0) # Prof. Green agreed with this corrected version and .fillna(False) avoids 'boolean value of NA is ambiguous' from nullable Float64 comparisons
funda['operprof'] = (funda['revt']-funda['cogs']-funda['xsga0']-funda['xint0'])/funda.groupby('gvkey')['ceq'].shift(1)
funda['ps'] = ((funda['ni']>0).fillna(False).astype(int)
              + (funda['oancf']>0).fillna(False).astype(int)
              + ((funda['ni']/funda['at']) > (funda.groupby('gvkey')['ni'].shift(1)/funda.groupby('gvkey')['at'].shift(1))).fillna(False).astype(int)
              + (funda['oancf']>funda['ni']).fillna(False).astype(int)
              + ((funda['dltt']/funda['at']) < (funda.groupby('gvkey')['dltt'].shift(1)/funda.groupby('gvkey')['at'].shift(1))).fillna(False).astype(int)
              + ((funda['act']/funda['lct']) > (funda.groupby('gvkey')['act'].shift(1)/funda.groupby('gvkey')['lct'].shift(1))).fillna(False).astype(int)
              + (((funda['sale']-funda['cogs'])/funda['sale']) > ((funda.groupby('gvkey')['sale'].shift(1)-funda.groupby('gvkey')['cogs'].shift(1))/funda.groupby('gvkey')['sale'].shift(1))).fillna(False).astype(int)
              + ((funda['sale']/funda['at']) > (funda.groupby('gvkey')['sale'].shift(1)/funda.groupby('gvkey')['at'].shift(1))).fillna(False).astype(int)
              + (funda['scstkc']==0).fillna(False).astype(int))

# prep for tb
funda['tr'] = np.select( # multi-condition np.where: first true condition wins, else default. use that instead of map or dframe because the conditions are year ranges, not exact values. .map() with a dictionary needs one key per exact year.
    [funda['fyear']<=1978,
     (funda['fyear']>=1979) & (funda['fyear']<=1986),
     funda['fyear']==1987,
     (funda['fyear']>=1988) & (funda['fyear']<=1992),
     funda['fyear']>=1993],
    [0.48, 0.46, 0.40, 0.34, 0.35],
    default=np.nan
)
funda['tb_1'] = np.where(funda['txfo'].isnull() | funda['txfed'].isnull(),
                          ((funda['txt']-funda['txdi'])/funda['tr'])/funda['ib'],
                          ((funda['txfo']+funda['txfed'])/funda['tr'])/funda['ib'])
tb1_override = ((((funda['txfo']+funda['txfed'])>0) | (funda['txt']>funda['txdi'])) & (funda['ib']<=0)).fillna(False)
funda['tb_1'] = np.where(tb1_override, 1, funda['tb_1']) # cond being tb1 override to be true

# prep for ms
funda['roa'] = funda['ni']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['cfroa'] = np.where(funda['oancf'].isnull(),
                          (funda['ib']+funda['dp'])/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2),
                          funda['oancf']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2))
funda['xrdint'] = funda['xrd']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['capxint'] = funda['capx']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)
funda['xadint'] = funda['xad']/((funda['at']+funda.groupby('gvkey')['at'].shift(1))/2)

# clean up for observations that do not have lagged observations to create variables (setting the first (three) row(s) to NA in the column where a lag was used)
req_vars = ['agr','invest','gma','chcsho','lgr','egr','chpm','chinv','hire','acc','pctacc','absacc','sgr',
            'pchsale_pchinvt','pchsale_pchrect','pchgm_pchsale','pchsale_pchxsga','pchcapx','ps','roa','cfroa', 
            'xrdint','capxint','xadint','divi','divo','grltnoa','rd','pchdepr','pchcurrat','pchquick','pchsaleinv','operprof']
funda.loc[funda['count']==1, req_vars] = np.nan # df.loc[rows, columns] - useful for some such as hire or ps because of .fillna()
funda.loc[funda['count']<3, ['chato','grcapx']] = np.nan # because using shift(2) BUT redundant because automatically put NaN. The lag puts NaN automatically unless the formula later replaces NaN

# other preparation steps for annual variables: industry adjustments
funda['chpmia'] = funda['chpm'] - funda.groupby(['sic2','fyear'])['chpm'].transform('mean') # computes a statistic for each group and repeats it on every row of that group
funda['chatoia'] = funda['chato'] - funda.groupby(['sic2','fyear'])['chato'].transform('mean')
funda['indsale'] = funda.groupby(['sic2','fyear'])['sale'].transform('sum')
funda['chempia'] = funda['hire'] - funda.groupby(['sic2','fyear'])['hire'].transform('mean')
funda['bm_ia'] = funda['bm'] - funda.groupby(['sic2','fyear'])['bm'].transform('mean')
funda['pchcapx_ia'] = funda['pchcapx'] - funda.groupby(['sic2','fyear'])['pchcapx'].transform('mean')
funda['tb'] = funda['tb_1'] - funda.groupby(['sic2','fyear'])['tb_1'].transform('mean')
funda['cfp_ia'] = funda['cfp'] - funda.groupby(['sic2','fyear'])['cfp'].transform('mean')
funda['mve_ia'] = funda['mve_f'] - funda.groupby(['sic2','fyear'])['mve_f'].transform('mean')

funda['herf'] = (funda['sale']/funda['indsale'])**2 # ** is the power operator
funda['herf'] = funda.groupby(['sic2','fyear'])['herf'].transform('sum')

# industry measures for ms
funda['md_roa'] = funda.groupby(['fyear','sic2'])['roa'].transform('median')
funda['md_cfroa'] = funda.groupby(['fyear','sic2'])['cfroa'].transform('median')
funda['md_xrdint'] = funda.groupby(['fyear','sic2'])['xrdint'].transform('median')
funda['md_capxint'] = funda.groupby(['fyear','sic2'])['capxint'].transform('median')
funda['md_xadint'] = funda.groupby(['fyear','sic2'])['xadint'].transform('median')

funda['m1'] = (funda['roa']>funda['md_roa']).fillna(False).astype(int)
funda['m2'] = (funda['cfroa']>funda['md_cfroa']).fillna(False).astype(int)
funda['m3'] = (funda['oancf']>funda['ni']).fillna(False).astype(int)
funda['m4'] = (funda['xrdint']>funda['md_xrdint']).fillna(False).astype(int)
funda['m5'] = (funda['capxint']>funda['md_capxint']).fillna(False).astype(int)
funda['m6'] = (funda['xadint']>funda['md_xadint']).fillna(False).astype(int) # still need to add m7

# add the cpi (using the revised from 2013 annual average CPI-U - so different for year > 2013 than the SAS code)

cpi = pd.DataFrame({ 
    'yr':[2016,2015,2014,2013,2012,2011,2010,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000,
          1999,1998,1997,1996,1995,1994,1993,1992,1991,1990,1989,1988,1987,1986,1985,1984,
          1983,1982,1981,1980,1979,1978,1977,1976,1975,1974],
    'cpi':[240.007,237.017,236.736,232.957,229.594,224.939,218.056,214.537,215.303,207.342,201.6,195.3,
       188.9,183.96,179.88,177.1,172.2,166.6,163.00,160.5,156.9,152.4,148.2,144.5,140.3,
       136.2,130.7,124.00,118.3,113.6,109.6,107.6,103.9,99.6,96.5,90.9,82.4,72.6,65.2,
       60.6,56.9,53.8,49.3]
})
funda = funda.merge(cpi, left_on='fyear', right_on='yr', how='left')

funda['avgat'] = (funda['at']+funda.groupby('gvkey')['at'].shift(1))/2 # scale at since big companies have larger asset

x = funda['xsga']/funda['cpi'] # SG&A scaled by CPI so that inflation doesnt make spending look like it grows
decay_pow = 0.85**(funda['count']-1) # 15% is the assumed yearly depreciation rate of organizational capital. It measures everything relative to the firm's first year in the data.
contrib = np.where(funda['count']==1, x/0.25, x/decay_pow) # x/0.25 is a guess at how much organizational capital the firm already had when it first appears in Compustat.
W = pd.Series(contrib, index=funda.index).groupby(funda['gvkey']).cumsum() # each row = its own value + all previous rows of the same firm
funda['orgcap'] = (W*decay_pow)/funda['avgat'] # Treat each year's real SG&A as an investment, let the accumulated stock depreciate 15% per year, scale by total assets
funda.loc[funda['count']==1, 'orgcap'] = np.nan 

In [6]:
# Add S&P long-term issuer credit rating (comp.adsprate) for credrat / credrat_dwn

adsprate = db.raw_sql('''
                      SELECT gvkey, datadate, splticrm
                      FROM comp.adsprate
                      WHERE datadate >='1975-01-01'
                      ''')

# splticrm being S&P long term issuer credit rating

adsprate['datadate'] = pd.to_datetime(adsprate['datadate']) 
adsprate['year'] = adsprate['datadate'].dt.year # since splticrm is monthly

funda['year'] = funda['datadate'].dt.year
funda = funda.merge(adsprate[['gvkey', 'year', 'splticrm']], on=['gvkey', 'year'], how='left') # link monthly to annual -> create monthly
funda = funda.sort_values(['gvkey', 'datadate']).drop_duplicates(subset=['gvkey', 'datadate']) # goes back for one row per year. kept month = first in adsprate's order from comp.adsprate. Why? Arbitrary?
funda = funda.drop(columns=['year'])

credrat_map = {
    'D': 1, 'C': 2, 'CC': 3, 'CCC-': 4, 'CCC': 5, 'CCC+': 6,
    'B-': 7, 'B': 8, 'B+': 9, 'BB-': 10, 'BB': 11, 'BB+': 12,
    'BBB-': 13, 'BBB': 14, 'BBB+': 15, 'A-': 16, 'A': 17, 'A+': 18,
    'AA-': 19, 'AA': 20, 'AA+': 21, 'AAA': 22,
}
funda['credrat'] = funda['splticrm'].map(credrat_map)

credrat_lag = funda.groupby('gvkey')['credrat'].shift(1)
funda['credrat_dwn'] = (funda['credrat'] < credrat_lag).fillna(False).astype(int)
funda.loc[funda['count'] == 1, 'credrat_dwn'] = 0

In [7]:
# Create merge with CRSP - get the data

ccmlink = db.raw_sql('''
                     SELECT gvkey, lpermno, linktype, linkdt, linkenddt
                     FROM crsp.ccmxpf_linktable
                     WHERE linktype in ('LU', 'LC', 'LD', 'LF', 'LN', 'LO', 'LS', 'LX')
                     ''')

# select only the reliable link type

In [8]:
# add permno to a temporary dataset

ccmlink['linkdt'] = pd.to_datetime(ccmlink['linkdt'])
ccmlink['linkenddt'] = pd.to_datetime(ccmlink['linkenddt'])

ccmlink = ccmlink[((ccmlink['linkdt'].dt.year <= 2015) | ccmlink['linkdt'].isna()) & # missing row in linkdt means always valid (from WRDS documentation)
          ((ccmlink['linkenddt'].dt.year >= 1950) | ccmlink['linkenddt'].isna())] # missing row in linkenddt means still valid (from WRDS documentation)

temp = funda.merge(ccmlink, on='gvkey', how='inner') # how='left' means keep 100% of rows from the left dataset (can also add some if the right has multiple match) and rows without a match get NaN in the new columns. how='inner': keeps only rows that match in both tables. Rows without a match are dropped.
temp = temp[
    ((temp['linkdt'] <= temp['datadate']) | temp['linkdt'].isna()) &
    ((temp['datadate'] <= temp['linkenddt']) | temp['linkenddt'].isna()) &
    temp['lpermno'].notna()]

temp = temp.rename(columns={'lpermno': 'permno'})
temp = temp.drop(columns=['linktype', 'linkdt', 'linkenddt'])
temp['permno'] = temp['permno'].astype('Int64')

In [9]:
# keep only common stocks and major exchanges

mseall = db.raw_sql('''
                    SELECT permno, date, exchcd, shrcd
                    FROM crsp.mseall
                    WHERE date >= '1972-01-01'
                    ''')

# dropped siccd from SAS pull since not needed later in the code

In [10]:
# screen for only NYSE, AMEX, NASDAQ, and common stock, then dedupe by (permno, exchcd, date)
mseall = mseall[(mseall['exchcd'].isin([1, 2, 3])) | (mseall['shrcd'].isin([10, 11, 12]))] # loose pre-filter (OR, incl. 12 = non-US incorporated), as in SAS; final filter in next cell keeps exchcd 1-3 AND shrcd 10/11. Why?
mseall = mseall.sort_values(['permno', 'exchcd', 'date'])
mseall = mseall.drop_duplicates(subset=['permno', 'exchcd', 'date'])

In [11]:
# exchange start/end dates per permno-exchcd segment
mseall['exchstdt'] = mseall.groupby(['permno', 'exchcd'])['date'].transform('min') # exchange start date
mseall['exchedt'] = mseall.groupby(['permno', 'exchcd'])['date'].transform('max') # exchange end date

# collapse to one row per permno-exchcd (keeps the earliest-dated row, same as SAS's nodupkey here)
mseall = mseall.sort_values(['permno', 'exchcd', 'date'])
mseall = mseall.drop_duplicates(subset=['permno', 'exchcd'])

# left join temp to mseall on permno, restricted to datadate falling within [exchstdt, exchedt]
temp = temp.merge(mseall, on='permno', how='left')
temp = temp[(temp['datadate'] >= temp['exchstdt']) & (temp['datadate'] <= temp['exchedt'])]

# keep only major-exchange common stocks (SAS comments out shrcd 12, keeping only 10/11)
temp = temp[(temp['exchcd'].isin([1, 2, 3])) & (temp['shrcd'].isin([10, 11])) & temp['permno'].notna()]
temp = temp.drop(columns=['shrcd', 'date', 'exchstdt', 'exchedt'])

# one row per gvkey-datadate
temp = temp.sort_values(['gvkey', 'datadate']).drop_duplicates(subset=['gvkey', 'datadate'])

In [12]:
# Finalize first Compustat data set
# This is most of the annual compustat variables plus a couple components that still need additional information

keep_cols = [
    'gvkey', 'permno', 'exchcd', 'datadate', 'fyear', 'sic2',
    'bm', 'cfp', 'ep', 'cashpr', 'dy', 'lev', 'sp', 'roic', 'rd_sale', 'rd_mve', 'agr', 'invest', 'gma',
    'chcsho', 'lgr', 'egr', 'chpm', 'chato', 'chinv', 'hire', 'acc', 'pctacc', 'absacc', 'age',
    'sgr', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchgm_pchsale', 'pchsale_pchxsga', 'pchcapx',
    'ps', 'divi', 'divo', 'securedind', 'secured', 'convind', 'grltnoa',
    'rd', 'chpmia', 'chatoia', 'chempia', 'bm_ia', 'pchcapx_ia', 'tb', 'cfp_ia', 'mve_ia', 'herf',
    'orgcap', 'm1', 'm2', 'm3', 'm4', 'm5', 'm6',
    'grcapx', 'depr', 'pchdepr', 'tang',
    'sin', 'mve_f', 'currat', 'pchcurrat', 'quick', 'pchquick',
    'salecash', 'salerec', 'saleinv', 'pchsaleinv', 'cashdebt', 'realestate', 'operprof',
]
temp = temp[keep_cols]

In [13]:
# Now align the annual Compustat variables in calendar month with the assumption that annual information is available with a lag of 6 months

msf = db.raw_sql('''
                 SELECT permno, date, ret, prc, shrout, vol
                 FROM crsp.msf
                 WHERE date >= '1972-01-01'
                 ''')

msf['date'] = pd.to_datetime(msf['date'])
msf['prc'] = msf['prc'].abs() 

# truncate datadate to its month, then jump n whole months
fy_month = temp['datadate'].values.astype('datetime64[M]') # dt.month gives only the month number (no year): 7 + 7 = 14, but we need Feb of next year. datetime64[M] keeps year + month, so month arithmetic rolls over correctly (2005-07 + 7 = 2006-02)
window_start = (fy_month + np.timedelta64(7, 'M')).astype('datetime64[ns]') # assumes at least a 6-month lag before investors could actually know the figures (fundamentals)
window_end = (fy_month + np.timedelta64(20, 'M')).astype('datetime64[ns]') # caps how long that fiscal year's data stays "current" before it should be replaced by the next fiscal year's filing. both are roughly a 12-13 month holding period

temp2 = temp.assign(window_start=window_start, window_end=window_end).merge(msf, on='permno', how='left') # window_start/end are plain arrays in temp's row order -> assign() attaches them by position, so each window sits next to the datadate it was computed from. creates way more rows - Each row after the merge carries the full accounting data of one fiscal year 
temp2 = temp2[(temp2['date'] >= temp2['window_start']) & (temp2['date'] < temp2['window_end'])] # keep months inside the fiscal year's window (7 to 19 months after fiscal year-end)
temp2 = temp2.drop(columns=['window_start', 'window_end'])

# The driver/backbone is msf of the whole datasaet (monthly returns) — every row in the output is anchored to a monthly observation, not a fiscal-year observation.
# The annual fundamentals (temp) get repeated/broadcast across however many consecutive months fall inside their [window_start, window_end) — so one fiscal-year's at, ceq, whatever accounting vars, get stamped onto ~12-13 monthly rows in a row. That's the "repetitive" part: the same annual number is attached to each month until the window rolls over to the next fiscal year.

In [14]:
# Included delisted returns in the monthly returns

mseall = db.raw_sql('''
                     SELECT permno, date, dlret, dlstcd, exchcd
                     FROM crsp.mseall
                     WHERE date >= '1972-01-01'
                     ''')

# dlstcd — delisting status code: A 3-digit code from CRSP explaining why a stock disappeared from the exchange
# dlret — delisting return: The return for the final partial period, from the last trading price to whatever (if anything) shareholders received at delisting
# CRSP sometimes just doesn't report it — the stock vanishes with no final price recorded, especially for bad-outcome delistings where there was no orderly final trade.
# if you just drop those stocks or treat missing dlret as 0, you systematically overstate returns, because the stocks that silently disappear tend to be the ones that went bankrupt/got liquidated — a real loss that just isn't recorded.
# for 'bad outcome' (500, 520–584) is will be negative return

mseall['date'] = pd.to_datetime(mseall['date'])

temp2 = temp2.drop(columns=['exchcd'])

temp2 = temp2.merge(mseall, on=['permno', 'date'], how='left')

delisted = temp2['dlret'].isna() & ((temp2['dlstcd'] == 500) | temp2['dlstcd'].between(520, 584)).fillna(False)  #true / false for the column temp2['dlret']. not like temp2[temp2['dlret'].isna()] 
# delisted '"was this stock delisted for a bad-outcome reason, AND does it have no recorded return that we need to fill in.'

temp2['dlret'] = temp2['dlret'].mask(delisted & temp2['exchcd'].isin([1, 2]), -0.35) # fixing NaN value for NYSE and AMEX
temp2['dlret'] = temp2['dlret'].mask(delisted & temp2['exchcd'].isin([3]), -0.55)  # fixing NaN value for NASDAQ
# .mask(cond, value) replaces entries where an external boolean condition is True (like np.where but only applying if TRUE)
# same as doing a['a']= np.where(cond, x,a['a'])

temp2['dlret'] = temp2['dlret'].mask(temp2['dlret'].notna() & (temp2['dlret'] < -1), -1.0) # Because a stock's total return can't logically go below -100% — the most you can lose owning equity is your entire investment 
temp2['dlret'] = temp2['dlret'].fillna(0)

temp2['ret'] = temp2['ret'] + temp2['dlret']
temp2['ret'] = temp2['ret'].mask(temp2['ret'].isna() & (temp2['dlret'] != 0), temp2['dlret']) # won't need dlret anymore

# proc sort by permno date descending datadate; proc sort nodupkey by permno date;
temp2 = temp2.sort_values(['permno', 'date', 'datadate'], ascending=[True, True, False]) # You want to keep only the row from the newer fiscal year (lastest avaiable info) (overlap in monthly accounting lag)
temp2 = temp2.drop_duplicates(subset=['permno', 'date'])

In [15]:
lag_prc = temp2.groupby('permno')['prc'].shift(1).abs()
lag_shrout = temp2.groupby('permno')['shrout'].shift(1)
temp2['mve_m'] = lag_prc * lag_shrout

temp2['mve'] = np.log(temp2['mve_m']) #  Taking the log compresses the huge range of market cap dramatically
temp2 = temp2[temp2.groupby('permno').cumcount() != 0]
# (SAS needs it because lag() would take the previous stock's value; in pandas it mainly keeps the sample identical to GHZ)

In [16]:
# now add in Compustat Quarterly and then add to the monthly aligned dataset

fundq = db.raw_sql('''
                 SELECT c.gvkey, fyearq,fqtr,datadate,rdq, sic,
                 ibq,saleq,txtq,revtq,cogsq,xsgaq,
                 atq,actq,cheq,lctq,dlcq,ppentq,
                 prccq, cshoq, ceqq,
                 seqq,pstkq,ltq,pstkrq 
                 FROM comp.company AS c 
                 JOIN comp.fundq AS f
                 ON f.gvkey = c.gvkey
                 WHERE datadate >='1975-01-01'
                 AND f.indfmt='INDL'
                 AND f.datafmt='STD'
                 AND f.popsrc='D'
                 AND f.consol='C'
                 ''')
# why two atq in the SAS sql pull?

fundq['datadate'] = pd.to_datetime(fundq['datadate'])
fundq['rdq'] = pd.to_datetime(fundq['rdq']) # rdq = earnings report date
fundq['sic2'] = fundq['sic'].str.slice(0,2) # convert to numbers later on?
fundq = fundq.drop(columns='sic') 
fundq['prccq'] = fundq['prccq'].abs()
fundq['mveq'] = fundq['prccq']*fundq['cshoq']
fundq = fundq.dropna(subset=['ibq']) # quarterly income before extraordinary items is the core earnings figure - a quarter without ibq usually means no real quarterly report was filed
fundq = fundq.sort_values(by=['gvkey','datadate']).drop_duplicates(subset=['gvkey', 'datadate'])

In [17]:
# create first set of quarterly compustat variables

g = fundq.groupby('gvkey')  # used for lag

# pstk = pstkrq if present, else pstkq
fundq['pstk'] = fundq['pstkrq'].fillna(fundq['pstkq'])

# scal: seqq -> else ceqq+pstk -> else atq-ltq
fundq['scal'] = np.where(fundq['seqq'].notna(), fundq['seqq'], fundq['ceqq'] + fundq['pstk'])  # seqq, else ceqq + pstk
cond_scal2 = fundq['seqq'].isna() & (fundq['ceqq'].isna() | fundq['pstk'].isna())
fundq['scal'] = fundq['scal'].mask(cond_scal2, fundq['atq'] - fundq['ltq'])                    # else atq - ltq

# chtx
fundq['chtx'] = (fundq['txtq'] - g['txtq'].shift(4)) / g['atq'].shift(4)

# roaq
fundq['roaq'] = fundq['ibq'] / g['atq'].shift(1)

# roeq
fundq['roeq'] = fundq['ibq'] / g['scal'].shift(1)

# rsup
fundq['rsup'] = (fundq['saleq'] - g['saleq'].shift(4)) / fundq['mveq']

# SAS treats missing as smaller than any number, so 'if saleq<=0' is also TRUE when saleq is missing.
# Python's NaN <= 0 is False, so add .isna() explicitly to match SAS: missing or <= 0 -> use 0.01
cond_bad_sale = fundq['saleq'].isna() | (fundq['saleq'] <= 0) # cond being saleq NA or negative or zero
den_sale = np.where(cond_bad_sale, 0.01, fundq['saleq']) # 0.01 to avoid dividing by zero

# sacc
actq_l1, cheq_l1, lctq_l1, dlcq_l1 = g['actq'].shift(1), g['cheq'].shift(1), g['lctq'].shift(1), g['dlcq'].shift(1) # called tuple unpacking
num_sacc = (fundq['actq'] - actq_l1 - (fundq['cheq'] - cheq_l1)) - ((fundq['lctq'] - lctq_l1) - (fundq['dlcq'] - dlcq_l1))
fundq['sacc'] = num_sacc / den_sale

# stdacc
fundq['stdacc'] = g['sacc'].transform(lambda s: s.rolling(16, min_periods=2).std()) # volatility of quarterly accruals: for each firm and quarter, the standard deviation of sacc over the last 16 quarters (4 years), including the current one.
# s.rolling(16, min_periods=2): a moving window of the current row plus the 15 before it. min_periods=2 means a result is produced once at least 2 non-missing values are in the window. Before that, it's NaN

# sgrvol
fundq['sgrvol'] = g['rsup'].transform(lambda s: s.rolling(15, min_periods=2).std()) # why 15 and not 16?

# roavol
fundq['roavol'] = g['roaq'].transform(lambda s: s.rolling(16, min_periods=2).std())

# scf (reuses sacc, which is already adjusted for the saleq<=0/missing case)
fundq['scf'] = fundq['ibq'] / den_sale - fundq['sacc']

# stdcf
fundq['stdcf'] = g['scf'].transform(lambda s: s.rolling(16, min_periods=2).std())

# cash
fundq['cash'] = fundq['cheq'] / fundq['atq']

# cinvest = corporate investment (how much the firm is investing this quarter compared with its average of the previous 3 quarters)
# EVERY denominator in the formula (including the lagged-sale ones) becomes 0.01.
ppent_l1, ppent_l2, ppent_l3, ppent_l4 = g['ppentq'].shift(1), g['ppentq'].shift(2), g['ppentq'].shift(3), g['ppentq'].shift(4)
sale_l1, sale_l2, sale_l3 = g['saleq'].shift(1), g['saleq'].shift(2), g['saleq'].shift(3)

den0 = den_sale
den1 = np.where(cond_bad_sale, 0.01, sale_l1)
den2 = np.where(cond_bad_sale, 0.01, sale_l2)
den3 = np.where(cond_bad_sale, 0.01, sale_l3)

term0 = (fundq['ppentq'] - ppent_l1) / den0
comp1 = (ppent_l1 - ppent_l2) / den1
comp2 = (ppent_l2 - ppent_l3) / den2
comp3 = (ppent_l3 - ppent_l4) / den3

mean_term = pd.concat([comp1, comp2, comp3], axis=1).mean(axis=1) # pd.concat([comp1, comp2, comp3], axis=1) puts the three Series next to each other as a 3-column table
fundq['cinvest'] = term0 - mean_term

# che (income change) -- used later for sue/nincr
fundq['che'] = fundq['ibq'] - g['ibq'].shift(4)

# nincr: length of the consecutive-earnings-increase streak, capped at 8 quarters of Income Before Extraordinary Items, Quarterly
ibq_lags = {}
for k in range(9):                        # k = 0, 1, 2, ..., 8
    ibq_lags[k] = g['ibq'].shift(k)       # ibq lagged k quarters, per firm

c = {}
for k in range(1, 9):
    c[k] = (ibq_lags[k-1] > ibq_lags[k]).fillna(False).astype(int)

fundq['nincr'] = (
    c[1]
    + c[1]*c[2]
    + c[1]*c[2]*c[3]
    + c[1]*c[2]*c[3]*c[4]
    + c[1]*c[2]*c[3]*c[4]*c[5]
    + c[1]*c[2]*c[3]*c[4]*c[5]*c[6]
    + c[1]*c[2]*c[3]*c[4]*c[5]*c[6]*c[7]
    + c[1]*c[2]*c[3]*c[4]*c[5]*c[6]*c[7]*c[8] # how many quarters IN A ROW earnings (ibq) have gone up
)

fundq['count'] = fundq.groupby('gvkey').cumcount() + 1

# first.gvkey (== count==1) -> null roaq/roeq
fundq.loc[fundq['count'] == 1, ['roaq', 'roeq']] = np.nan # because of lag. Even though already NaN, so the line is redundant. roaq = ibq / lag1(atq), and in the first quarter lag1 is NaN, so the result is NaN

# count<5 -> null chtx/che/cinvest (these use up to lag(4))
fundq.loc[fundq['count'] < 5, ['chtx', 'che', 'cinvest']] = np.nan # same

# count<17 -> null the rolling-std measures (these need up to a 16-quarter window)
fundq.loc[fundq['count'] < 17, ['stdacc', 'stdcf', 'sgrvol', 'roavol']] = np.nan # Alternatively, you could use min_periods=16 and drop this line.

In [18]:
# finally finish Mohanram score components

fundq['md_roavol'] = fundq.groupby(['fyearq', 'fqtr', 'sic2'])['roavol'].transform('median') # fqtr: fiscal quarter field (1-4)
fundq['md_sgrvol'] = fundq.groupby(['fyearq', 'fqtr', 'sic2'])['sgrvol'].transform('median')

# SAS treats missing numeric as -infinity in comparisons: "roavol<md_roavol" is TRUE
# whenever roavol is missing and md_roavol isn't -- not just when roavol is a smaller real number.
m7_cond = (fundq['roavol'] < fundq['md_roavol']) | (fundq['roavol'].isna() & fundq['md_roavol'].notna())
fundq['m7'] = m7_cond.astype(int)

m8_cond = (fundq['sgrvol'] < fundq['md_sgrvol']) | (fundq['sgrvol'].isna() & fundq['md_sgrvol'].notna())
fundq['m8'] = m8_cond.astype(int)

fundq = fundq.dropna(subset=['rdq'])

# adding permno via ccmlink

fundq = fundq.merge(ccmlink, on='gvkey', how='inner')
fundq = fundq[
    ((fundq['linkdt'] <= fundq['datadate']) | fundq['linkdt'].isna()) &
    ((fundq['datadate'] <= fundq['linkenddt']) | fundq['linkenddt'].isna()) &
    fundq['lpermno'].notna()
]
fundq = fundq.rename(columns={'lpermno': 'permno'}).drop(columns=['linktype', 'linkdt', 'linkenddt'])
fundq['permno'] = fundq['permno'].astype('Int64')

In [19]:
# get daily stock informations

dsf = db.raw_sql('''
                 SELECT permno, date, vol, ret
                 FROM crsp.dsf
                 WHERE date BETWEEN '1974-11-01' AND '2016-12-31'
                 ''')

dsf['date'] = pd.to_datetime(dsf['date'])

In [20]:
# Some of the RPS require daily CRSP data in conjunction with Compustat quarterly, so add daily CRSP info to create these RPS ##
# For abnormal trading volume and returns around earings announcements

q = fundq[['permno', 'datadate', 'rdq']].dropna(subset=['permno', 'rdq']).drop_duplicates() # keep one permno/quarter obs

def expand_window(q, start, end):
    # one row per weekday from rdq+start to rdq+end, same days as the old BDay filter
    parts = [q.assign(date=q['rdq'] + pd.tseries.offsets.BDay(k)) for k in range(start, end + 1)]
    return pd.concat(parts, ignore_index=True).drop_duplicates()

# --- avgvol: mean daily volume in the [-30, -10] weekday window before rdq ---
avgvol_win = expand_window(q, -30, -10).merge(dsf[['permno', 'date', 'vol']], on=['permno', 'date'], how='inner')
avgvol = avgvol_win.groupby(['permno', 'datadate', 'rdq'])['vol'].mean().reset_index(name='avgvol')
fundq = fundq.merge(avgvol, on=['permno', 'datadate', 'rdq'], how='left')

# --- aeavol and ear: mean volume and summed return in the [-1, +1] weekday window around rdq ---
ear_win = expand_window(q, -1, 1).merge(dsf[['permno', 'date', 'vol', 'ret']], on=['permno', 'date'], how='inner')
ear_agg = ear_win.groupby(['permno', 'datadate', 'rdq']).agg(
    vol_mean=('vol', 'mean'),
    ear=('ret', 'sum'),
).reset_index()
ear_agg = ear_agg.merge(avgvol, on=['permno', 'datadate', 'rdq'], how='left')
ear_agg['aeavol'] = (ear_agg['vol_mean'] - ear_agg['avgvol']) / ear_agg['avgvol']

fundq = fundq.merge(ear_agg[['permno', 'datadate', 'rdq', 'ear', 'aeavol']], on=['permno', 'datadate', 'rdq'], how='left')


In [21]:
# First Compustat quarterly data set

fundq = fundq[['gvkey', 'permno', 'datadate', 'rdq',
               'chtx', 'roaq', 'rsup', 'stdacc', 'stdcf', 'sgrvol', 'roavol',
               'cash', 'cinvest', 'nincr', 'aeavol', 'ear', 'm7', 'm8', 'prccq', 'roeq']]

In [22]:
# add quarterly compustat data to monthly returns and annual compustat data

temp2 = temp2.drop(columns=['datadate']) # drop annual datadate to merge quarterly date

date_month = temp2['date'].values.astype('datetime64[M]')
window_start = (date_month + np.timedelta64(-10, 'M')).astype('datetime64[ns]')
window_end = (date_month + np.timedelta64(-5, 'M')).astype('datetime64[ns]')

# Together, [-10, -5] is a rolling 5-month-wide eligibility window: a quarter becomes usable once it's at least 5 months old (public by then), and stays usable for 5 more months (until month 10) before expiring — matching the fact that firms file 4x/year, so a ~5-month window almost always guarantees exactly one "current" quarter is in scope at any time. That's narrower than the annual block's 13-month window (+7 to +20) for the same reason: quarterly data turns over 4x as often as annual data, so the eligibility window needs to be proportionally shorter.

# merge_asof: for each month, take the most recent quarter with datadate <= window_end.
# Same result as merging on permno, filtering to the window and keeping the latest quarter,
# but without pairing every month with every quarter of the firm (that ran out of memory).
left = temp2.assign(window_start=window_start, window_end=window_end).dropna(subset=['permno']).sort_values('window_end')
right = fundq.dropna(subset=['permno', 'datadate']).sort_values('datadate')
right['datadate'] = right['datadate'].astype('datetime64[ns]')


temp3 = pd.merge_asof(left, right, left_on='window_end', right_on='datadate', by='permno', direction='backward')
temp3 = temp3[temp3['datadate'] >= temp3['window_start']] # quarter must also be inside the window, not older than -10 months
temp3 = temp3.drop(columns=['window_start', 'window_end'])
temp3 = temp3.sort_values(['permno', 'date']).reset_index(drop=True)

# eamonth = 1 if the firm announced earnings (rdq) in this calendar month
ea = fundq.loc[fundq['permno'].notna() & fundq['rdq'].notna(), ['permno', 'rdq']]
ea = ea.assign(ym=ea['rdq'].dt.to_period('M'))[['permno', 'ym']].drop_duplicates() # one row per permno-month, so the merge can't duplicate rows
ea['eamonth'] = 1

temp3['ym'] = temp3['date'].dt.to_period('M')
temp3 = temp3.merge(ea, on=['permno', 'ym'], how='left').drop(columns=['ym'])
temp3['eamonth'] = temp3['eamonth'].fillna(0).astype(int) # for clean 0/1 integer

temp3['ms'] = (temp3['m1'] + temp3['m2'] + temp3['m3'] + temp3['m4']
               + temp3['m5'] + temp3['m6'] + temp3['m7'] + temp3['m8'])
temp3 = temp3.drop(columns=['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8'])

In [23]:
# There are some other variables that are based on monthly-CRSP information (already in the dataset from monthly CRSP) 
# create those variables plus a couple of others

temp3 = temp3[temp3['ret'].notna()].sort_values(['permno', 'date']).reset_index(drop=True)
temp3['count'] = temp3.groupby('permno').cumcount() + 1

g = temp3.groupby('permno')

temp3['mom1m'] = g['ret'].shift(1)

temp3['mom6m'] = (
    (1 + g['ret'].shift(2)) * (1 + g['ret'].shift(3)) * (1 + g['ret'].shift(4))
    * (1 + g['ret'].shift(5)) * (1 + g['ret'].shift(6))
) - 1

temp3['mom12m'] = (
    (1 + g['ret'].shift(2)) * (1 + g['ret'].shift(3)) * (1 + g['ret'].shift(4))
    * (1 + g['ret'].shift(5)) * (1 + g['ret'].shift(6)) * (1 + g['ret'].shift(7))
    * (1 + g['ret'].shift(8)) * (1 + g['ret'].shift(9)) * (1 + g['ret'].shift(10))
    * (1 + g['ret'].shift(11)) * (1 + g['ret'].shift(12))
) - 1

temp3['mom36m'] = 1.0
for k in range(13, 37):
    temp3['mom36m'] = temp3['mom36m'] * (1 + g['ret'].shift(k))
temp3['mom36m'] = temp3['mom36m'] - 1

temp3['dolvol'] = np.log(g['vol'].shift(2) * g['prc'].shift(2))

temp3['chmom'] = (
    ((1 + g['ret'].shift(1)) * (1 + g['ret'].shift(2)) * (1 + g['ret'].shift(3))
     * (1 + g['ret'].shift(4)) * (1 + g['ret'].shift(5)) * (1 + g['ret'].shift(6))) - 1
) - (
    ((1 + g['ret'].shift(7)) * (1 + g['ret'].shift(8)) * (1 + g['ret'].shift(9))
     * (1 + g['ret'].shift(10)) * (1 + g['ret'].shift(11)) * (1 + g['ret'].shift(12))) - 1
)

turn_lags = pd.concat([g['vol'].shift(1), g['vol'].shift(2), g['vol'].shift(3)], axis=1)
temp3['turn'] = turn_lags.mean(axis=1) / temp3['shrout'] # This builds a small 3-column DataFrame of the three lagged volumes side by side, so they can be averaged row-wise (axis=1) in the next line.

temp3.loc[temp3['count'] == 1, 'mom1m'] = np.nan
temp3.loc[temp3['count'] < 13, ['mom12m', 'chmom']] = np.nan
temp3.loc[temp3['count'] < 7, 'mom6m'] = np.nan
temp3.loc[temp3['count'] < 37, 'mom36m'] = np.nan
temp3.loc[temp3['count'] < 3, 'dolvol'] = np.nan
temp3.loc[temp3['count'] < 4, 'turn'] = np.nan

temp3['indmom'] = temp3.groupby(['sic2', 'date'])['mom12m'].transform('mean')

/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/pandas/core/arrays/masked.py:902: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs2, **kwargs)


In [ ]:
# finally, a few more directly from daily CRSP to create monthly variables
# Done one year at a time so the daily data never gets too big for memory

if os.path.exists('dcrsp.parquet'):
    dcrsp = pd.read_parquet('dcrsp.parquet')       # already downloaded before: load in seconds
else:
    parts = []
    for yr in range(1970, 2017):
        for attempt in range(3):                   # retry if the connection drops
            try:
                d = db.raw_sql(f'''
                    SELECT permno, date, ret, prc, vol, shrout, askhi, bidlo
                    FROM crsp.dsf
                    WHERE date BETWEEN '{yr}-01-01' AND '{yr}-12-31'
                    ''', chunksize=None,
                    dtype={'permno': 'float64', 'ret': 'float64', 'prc': 'float64', 'vol': 'float64',
                           'shrout': 'float64', 'askhi': 'float64', 'bidlo': 'float64'})
                break
            except Exception:
                print(f'\n{yr}: connection lost, reconnecting...')
                db = wrds.Connection(wrds_username='hcdb')

        d['date'] = pd.to_datetime(d['date'])
        d['dolvol_log'] = np.log((d['prc'] * d['vol']).abs()).replace(-np.inf, np.nan)
        d['baspread_daily'] = (d['askhi'] - d['bidlo']) / ((d['askhi'] + d['bidlo']) / 2)
        d['turn_daily'] = d['vol'] / d['shrout']
        d['ill_daily'] = (d['ret'].abs() / (d['prc'].abs() * d['vol'])).replace([np.inf, -np.inf], np.nan)
        d['zero_vol'] = (d['vol'] == 0).astype(int)
        d['yr'] = d['date'].dt.year
        d['month'] = d['date'].dt.month

        parts.append(d.groupby(['permno', 'yr', 'month']).agg(
            maxret=('ret', 'max'),
            retvol=('ret', 'std'),
            baspread=('baspread_daily', 'mean'),
            std_dolvol=('dolvol_log', 'std'),
            std_turn=('turn_daily', 'std'),
            ill=('ill_daily', 'mean'),
            countzero=('zero_vol', 'sum'),
            ndays=('permno', 'count'),
            turn=('turn_daily', 'sum'),
        ).reset_index())
        print(yr, end=' ')

    dcrsp = pd.concat(parts, ignore_index=True)
    dcrsp.to_parquet('dcrsp.parquet')              # save so this download never has to run again


/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:438: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


1970 

/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:438: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


1971 

/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:438: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


1972 

... (85 outputs left out)

2015 

/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:438: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


2016 

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip, conda, or your preferred package management tool to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip, conda, or your preferred package management tool to install the fastparquet package.

In [25]:
dcrsp.to_parquet('dcrsp.parquet')              # save so this download never has to run again

In [26]:
dcrsp['zerotrade'] = (dcrsp['countzero'] + ((1 / dcrsp['turn']) / 480000)) * 21 / dcrsp['ndays'] # since 21 trading days a months

lag_date = temp3['date'] - pd.DateOffset(months=1)
temp3['_lag_yr'] = lag_date.dt.year
temp3['_lag_month'] = lag_date.dt.month

temp3 = temp3.merge(
    dcrsp[['permno', 'yr', 'month', 'maxret', 'retvol', 'baspread', 'std_dolvol', 'std_turn', 'ill', 'zerotrade']],
    left_on=['permno', '_lag_yr', '_lag_month'], right_on=['permno', 'yr', 'month'], how='left',
)
temp3 = temp3.drop(columns=['_lag_yr', '_lag_month', 'yr', 'month'])

In [27]:
# weekly returns, built one year at a time (same reason as the daily loop: memory)
# intnx('WEEK', date, 0, 'end') -- the Saturday that ends date's Sun-Sat week
# compound daily returns within each stock-week: exp(sum(log(1+ret))) - 1

if os.path.exists('wk.parquet'):
    wk = pd.read_parquet('wk.parquet')             # already built before: load in seconds
else:
    wparts = []
    for yr in range(1974, 2017):  # start 1974 so the first 1975 week (Dec 29 - Jan 4) is complete
        for attempt in range(3):                   # retry if the connection drops
            try:
                d = db.raw_sql(f'''
                    SELECT permno, date, ret
                    FROM crsp.dsf
                    WHERE date BETWEEN '{yr}-01-01' AND '{yr}-12-31'
                    ''', chunksize=None, dtype={'permno': 'float64', 'ret': 'float64'})
                break
            except Exception:
                print(f'\n{yr}: connection lost, reconnecting...')
                db = wrds.Connection(wrds_username='hcdb')

        d['date'] = pd.to_datetime(d['date'])
        d['wkdt'] = d['date'] + pd.to_timedelta((5 - d['date'].dt.dayofweek) % 7, unit='D')
        d['log1ret'] = np.log(1 + d['ret'])  # cant log neg values so need +1
        wparts.append(d.groupby(['permno', 'wkdt'])['log1ret'].sum().reset_index())
        print(yr, end=' ')

    # a week crossing New Year shows up in two years, so add the pieces together again
    wk = pd.concat(wparts).groupby(['permno', 'wkdt'])['log1ret'].sum().reset_index(name='sum_log1ret')
    wk['wkret'] = np.exp(wk['sum_log1ret']) - 1
    wk = wk.drop(columns=['sum_log1ret'])
    wk = wk[wk['wkdt'] >= '1975-01-01']

    # equal-weighted weekly market return: average wkret across all stocks in that same week
    wk['ewret'] = wk.groupby('wkdt')['wkret'].transform('mean')

    wk = wk.dropna(subset=['wkret', 'ewret'])
    wk.to_parquet('wk.parquet')                    # save so this download never has to run again


/var/folders/1s/4zlwcvts2ss_pqsslh1jyqvr0000gn/T/ipykernel_60458/1548138018.py:23: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d['wkdt'] = d['date'] + pd.to_timedelta((5 - d['date'].dt.dayofweek) % 7, unit='D')


1974 

/var/folders/1s/4zlwcvts2ss_pqsslh1jyqvr0000gn/T/ipykernel_60458/1548138018.py:23: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d['wkdt'] = d['date'] + pd.to_timedelta((5 - d['date'].dt.dayofweek) % 7, unit='D')


1975 

/var/folders/1s/4zlwcvts2ss_pqsslh1jyqvr0000gn/T/ipykernel_60458/1548138018.py:23: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d['wkdt'] = d['date'] + pd.to_timedelta((5 - d['date'].dt.dayofweek) % 7, unit='D')


1976 

... (76 outputs left out)

/var/folders/1s/4zlwcvts2ss_pqsslh1jyqvr0000gn/T/ipykernel_60458/1548138018.py:23: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d['wkdt'] = d['date'] + pd.to_timedelta((5 - d['date'].dt.dayofweek) % 7, unit='D')


2015 

/var/folders/1s/4zlwcvts2ss_pqsslh1jyqvr0000gn/T/ipykernel_60458/1548138018.py:23: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d['wkdt'] = d['date'] + pd.to_timedelta((5 - d['date'].dt.dayofweek) % 7, unit='D')


2016 

In [28]:
# Beta, idiovol and pricedelay: one firm at a time.
# For each firm-month, slice that firm's weekly data to the [-36, -1] month window and run
# the two regressions there. Same results as building the full fan-out table, without the memory.

def ols(y, X):
    # least squares; returns coefficients, adjusted R^2 and residuals (same formulas as statsmodels)
    coef = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ coef
    n, k = X.shape
    r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return coef, 1 - (1 - r2) * (n - 1) / (n - k), resid

wk = wk.sort_values(['permno', 'wkdt']).reset_index(drop=True)
wk_permno = wk['permno'].values
wk_date, wk_ret, wk_mkt = wk['wkdt'].values, wk['wkret'].values, wk['ewret'].values

# unique firm-months that need a beta estimate, with their 3-year window ending 1 month before
lst = temp3.loc[temp3['permno'].notna() & temp3['date'].notna(), ['permno', 'date']].drop_duplicates()
lst['window_start'] = lst['date'] - pd.DateOffset(months=36)
lst['window_end'] = lst['date'] - pd.DateOffset(months=1)

rows = []
for i, (p, months) in enumerate(lst.groupby('permno')):
    # this firm's weekly rows (wk is sorted by permno, so they sit next to each other)
    a, b = np.searchsorted(wk_permno, p, 'left'), np.searchsorted(wk_permno, p, 'right')
    t, r, m = wk_date[a:b], wk_ret[a:b], wk_mkt[a:b]

    # each month's window as positions in t: wkdt >= window_start and wkdt <= window_end
    starts = np.searchsorted(t, months['window_start'].values, 'left')
    ends = np.searchsorted(t, months['window_end'].values, 'right')

    for date, s, e in zip(months['date'].values, starts, ends):
        n = e - s
        if n < 52:  # require at least 52 weeks (~1 year) of history in that window
            continue
        y, x = r[s:e], m[s:e]

        # Regression 1: wkret = a + b*ewmkt
        c1, rsq1, _ = ols(y, np.column_stack([np.ones(n), x]))

        # Regression 2: wkret = a + b0*ewmkt + b1*l1 + ... + b4*l4, lags within the window,
        # so the first 4 weeks have no lags and drop out (same as SAS's "if count<5" block)
        X2 = np.column_stack([np.ones(n - 4), x[4:], x[3:-1], x[2:-2], x[1:-3], x[:-4]])
        _, adjrsq2, resid2 = ols(y[4:], X2)

        rows.append((p, date, c1[1], rsq1, adjrsq2, resid2.std(ddof=1)))

    if i % 2000 == 0:
        print(i, end=' ')  # progress: number of firms done

beta_out = pd.DataFrame(rows, columns=['permno', 'date', 'beta', 'rsq1', 'adjrsq2', 'idiovol'])
beta_out['betasq'] = beta_out['beta'] ** 2
beta_out['pricedelay'] = 1 - beta_out['rsq1'] / beta_out['adjrsq2']
beta_out = beta_out[['permno', 'date', 'beta', 'betasq', 'rsq1', 'pricedelay', 'idiovol']]


0 2000 

/var/folders/1s/4zlwcvts2ss_pqsslh1jyqvr0000gn/T/ipykernel_60458/2890243026.py:10: RuntimeWarning: invalid value encountered in scalar divide
  r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()


4000 6000 8000 10000 12000 14000 16000 

In [29]:
temp4 = temp3.merge(beta_out, on=['permno', 'date'], how='left')
temp4 = temp4[temp4['date'].dt.year >= 1980]
temp4 = temp4.sort_values(['permno', 'date']).drop_duplicates(['permno', 'date']).reset_index(drop=True)


In [30]:
temp = temp4.dropna(subset=['mve', 'mom1m', 'bm']).copy()

In [31]:
# this is for those bounded below at zero but may have large positive outliers

hitrim = ['betasq', 'mve', 'dy', 'lev', 'baspread', 'depr', 'sp', 'turn', 'dolvol', 'std_dolvol',
          'std_turn', 'idiovol', 'roavol', 'ill', 'age', 'rd_sale', 'rd_mve',
          'retvol', 'zerotrade', 'stdcf', 'tang', 'absacc', 'stdacc', 'cash', 'orgcap', 'salecash',
          'salerec', 'saleinv', 'pchsaleinv', 'cashdebt', 'realestate', 'secured']

# this is for those that may have large positive or negative outliers

hilotrim = ['beta', 'ep', 'mom12m', 'mom1m', 'mom6m', 'mom36m', 'indmom', 'agr',
            'maxret', 'bm', 'currat', 'pchcurrat', 'quick', 'pchquick', 'pchdepr', 'sgr',
            'chempia', 'acc', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchcapx_ia', 'pchgm_pchsale',
            'pchsale_pchxsga', 'mve_ia', 'cfp_ia', 'bm_ia', 'chinv', 'grltnoa', 'cinvest',
            'tb', 'cfp', 'lgr', 'egr', 'pricedelay', 'grcapx', 'chmom', 'roic', 'aeavol', 'chcsho',
            'chpmia', 'chatoia', 'ear', 'rsup', 'hire', 'cashpr', 'roaq',
            'roeq', 'invest', 'chtx', 'pctacc', 'gma', 'operprof']


# Some of these are not continuous, they are dummy variables so they are excluded 
# from the outlier issue

In [32]:
# winsorize hitrim vars: upper tail only (bounded below at 0)
temp2 = temp.copy()
p99 = temp2.groupby('date')[hitrim].transform(lambda s: s.quantile(0.99))

for col in hitrim:
    temp2[col] = temp2[col].clip(upper=p99[col])          # .clip is NaN-safe: leaves missing values untouched
    temp2.loc[p99[col].isna(), col] = np.nan              # null whole var for dates where p99 itself is undefined

# winsorize hilotrim vars: both tails 
p1 = temp2.groupby('date')[hilotrim].transform(lambda s: s.quantile(0.01))
p99b = temp2.groupby('date')[hilotrim].transform(lambda s: s.quantile(0.99))

for col in hilotrim:
    temp2[col] = temp2[col].clip(lower=p1[col], upper=p99b[col])
    temp2.loc[p1[col].isna(), col] = np.nan 

temp2 = temp2.sort_values('date').reset_index(drop=True)

/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/Users/emilefernandes/Desktop/PhD Application /Coding/Python/GKX-ML-Asset-Pricing-Python/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/Users/emilefernandes/Desktop/PhD Application /Coding/Py

In [34]:
temp2.to_parquet('rpsdata_python.parquet')